In [27]:
# Evaluate and tune Gaussian Process hyperparameters
# 1 GPR model for each of the 3 target properties 
# Compare kernel settings using leave-one-out cross-validation (R²)
# Visualize 1D slices of the full GP by varying one ingredient while holding
# others constant, sampling 50 posterior functions

In [28]:
# import data

from pathlib import Path
import pandas as pd

# Load the compiled data
project_root = Path.cwd().resolve().parent if 'notebooks' in str(Path.cwd()) else Path.cwd().resolve()
excel_path = project_root / 'data' / 'raw' / 'Compiled Data.xlsx'

df = pd.read_excel(excel_path)

print()

In [29]:
# Show the 25 original formulas and their three measured targets from df.
formula_cols = ["TEM", "Joncryl", "NC513", "PEG-DE", "NGDE"]
measured_target_cols = [
    "Avg_Density_g_cm3",
    "24h_avg_pct absorption",
    "24h_avg_pct_retention",
]
raw_data_summary = df[formula_cols + measured_target_cols].copy()
raw_data_summary[formula_cols] = raw_data_summary[formula_cols].fillna(0.0)
raw_data_summary[measured_target_cols] = raw_data_summary[measured_target_cols].apply(
    pd.to_numeric, errors="raise"
)

assert len(raw_data_summary) == 25, (
    f"Expected the 25 original samples; found {len(raw_data_summary)}."
)
display(raw_data_summary)

,TEM,Joncryl,NC513,PEG-DE,NGDE,Avg_Density_g_cm3,24h_avg_pct absorption,24h_avg_pct_retention
0,8,0.0,0.0,0.0,0.0,0.192000,113.616943,31.770000
1,8,0.7,0.0,0.0,0.0,0.195000,103.817309,33.120000
2,8,1.0,0.0,0.0,0.0,0.159000,129.464292,36.250000
3,8,1.5,0.0,0.0,0.0,0.159933,120.749405,25.330000
4,8,2.0,0.0,0.0,0.0,0.201000,113.871581,26.580000
5,8,3.0,0.0,0.0,0.0,0.202000,111.044347,27.400000
6,5,0.0,0.0,0.0,0.0,0.298930,56.958538,36.614852
7,5,0.0,0.7,0.0,0.0,0.360035,57.115160,44.587460
8,5,0.0,1.0,0.0,0.0,0.355557,44.308895,43.766577
9,5,0.0,2.0,0.0,0.0,0.358055,43.326301,44.186536


In [ ]:
# Compare GPR kernel settings with leave-one-out cross-validation, then fit
# one selected Gaussian Process model for each target property.

import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, RBF, WhiteKernel
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

feature_cols = ["TEM", "Joncryl", "NC513", "PEG-DE", "NGDE"]
target_cols = [
    "Avg_Density_g_cm3",
    "24h_avg_pct absorption",
    "24h_avg_pct_retention",
]
target_labels = {
    "Avg_Density_g_cm3": "Average density (g/cm?)",
    "24h_avg_pct absorption": "24-hour absorption (%)",
    "24h_avg_pct_retention": "24-hour retention (%)",
}

model_data = df[feature_cols + target_cols].copy()
model_data[feature_cols] = model_data[feature_cols].fillna(0.0)
model_data[target_cols] = model_data[target_cols].apply(pd.to_numeric, errors="raise")
if model_data[target_cols].isna().any().any():
    raise ValueError("Target measurements contain missing values.")

X = model_data[feature_cols]
loo = LeaveOneOut()

# Raise the minimum length scale to encourage smoother fits; optimize within each fold.
def make_kernel_candidates(min_length_scale=1.0):
    return {
        "RBF": lambda: ConstantKernel(1.0, (1e-3, 1e3))
        * RBF(length_scale=np.ones(len(feature_cols)), length_scale_bounds=(min_length_scale, 1e5))
        + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-8, 1e1)),
        "Mat?rn 1.5": lambda: ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(length_scale=np.ones(len(feature_cols)), length_scale_bounds=(min_length_scale, 1e5), nu=1.5)
        + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-8, 1e1)),
        "Mat?rn 2.5": lambda: ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(length_scale=np.ones(len(feature_cols)), length_scale_bounds=(min_length_scale, 1e5), nu=2.5)
        + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-8, 1e1)),
    }


def make_gpr(kernel, n_restarts_optimizer=0):
    return make_pipeline(
        StandardScaler(),
        GaussianProcessRegressor(
            kernel=kernel,
            normalize_y=True,
            n_restarts_optimizer=n_restarts_optimizer,
            random_state=42,
        ),
    )


cv_rows = []
gpr_models = {}
best_cv_predictions = {}

for target in target_cols:
    y = model_data[target]
    min_length_scale = {
        "Avg_Density_g_cm3": 1.0,
        "24h_avg_pct absorption": 3.0,
        "24h_avg_pct_retention": 1.0,
    }[target]
    kernel_candidates = make_kernel_candidates(min_length_scale)
    predictions_by_kernel = {}
    for kernel_name, kernel_factory in kernel_candidates.items():
        oof_prediction = np.empty(len(model_data), dtype=float)
        for train_indices, test_indices in loo.split(X):
            model = make_gpr(kernel_factory())
            model.fit(X.iloc[train_indices], y.iloc[train_indices])
            oof_prediction[test_indices] = model.predict(X.iloc[test_indices])
        predictions_by_kernel[kernel_name] = oof_prediction
        cv_rows.append({
            "Property": target_labels[target],
            "Kernel": kernel_name,
            "LOOCV R?": r2_score(y, oof_prediction),
            "LOOCV MAE": mean_absolute_error(y, oof_prediction),
            "LOOCV RMSE": np.sqrt(mean_squared_error(y, oof_prediction)),
        })

    target_rows = [row for row in cv_rows if row["Property"] == target_labels[target]]
    best_result = max(target_rows, key=lambda row: row["LOOCV R?"])
    best_cv_predictions[target] = predictions_by_kernel[best_result["Kernel"]]
    gpr_models[target] = make_gpr(kernel_candidates[best_result["Kernel"]]()).fit(X, y)

# Save the fitted models as a dictionary keyed by the target column name.
import joblib

models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / "gpr_models.joblib"
joblib.dump(gpr_models, model_path)
print(f"Saved {len(gpr_models)} target-specific GPR models to {model_path}")

cv_results = pd.DataFrame(cv_rows).sort_values(
    ["Property", "LOOCV R?"], ascending=[True, False]
).reset_index(drop=True)
print("LOOCV comparison (higher R? is better):")
display(cv_results.round(4))

best_model_summary = []
for target in target_cols:
    target_results = cv_results.loc[cv_results["Property"] == target_labels[target]]
    selected = target_results.loc[target_results["LOOCV R?"].idxmax()]
    fitted_gpr = gpr_models[target].named_steps["gaussianprocessregressor"]
    best_model_summary.append({
        "Property": target_labels[target],
        "Selected kernel": selected["Kernel"],
        "Minimum length-scale bound": {
            "Avg_Density_g_cm3": 1.0,
            "24h_avg_pct absorption": 3.0,
            "24h_avg_pct_retention": 1.0,
        }[target],
        "LOOCV R?": selected["LOOCV R?"],
        "Optimized kernel (all samples)": str(fitted_gpr.kernel_),
    })
print("Selected model for each property, refit on all samples:")
display(pd.DataFrame(best_model_summary))


In [ ]:
# Plot posterior slices anchored at two measured formulas per target.
# Faint points are all measurements; stars mark the formulas anchoring the slices.

import matplotlib.pyplot as plt

n_grid = 100
n_posterior_samples = 2
rng = np.random.default_rng(42)
fig, axes = plt.subplots(
    len(feature_cols), len(target_cols),
    figsize=(16, 3.2 * len(feature_cols)),
    squeeze=False,
)

for row, feature in enumerate(feature_cols):
    feature_min = X[feature].min()
    feature_max = X[feature].max()
    if feature_min == feature_max:
        raise ValueError(f"Cannot plot a slice for constant feature {feature!r}.")
    if feature == "TEM":
        feature_grid = np.linspace(feature_min, feature_max, n_grid)
    else:
        feature_grid = np.linspace(0.0, 6.0, n_grid)

    for col, target in enumerate(target_cols):
        axis = axes[row, col]
        pipeline = gpr_models[target]
        gpr = pipeline.named_steps["gaussianprocessregressor"]
        anchor_indices = list(dict.fromkeys([
            model_data[target].idxmin(),
            model_data[target].idxmax(),
        ]))
        anchor_colors = ["navy", "darkgreen"]

        for sample_index, (anchor_index, color) in enumerate(zip(anchor_indices, anchor_colors)):
            baseline_formula = X.loc[anchor_index]
            anchor_x = X.loc[anchor_index, feature]
            slice_grid = np.unique(np.append(feature_grid, anchor_x))
            grid_data = pd.DataFrame(
                np.repeat(baseline_formula.to_numpy()[None, :], len(slice_grid), axis=0),
                columns=feature_cols,
            )
            grid_data[feature] = slice_grid
            scaled_grid = pipeline.named_steps["standardscaler"].transform(grid_data)
            mean, covariance = gpr.predict(scaled_grid, return_cov=True)
            # GPR rescales covariance to target units; convert noise variance to match.
            target_scale = model_data[target].std(ddof=0)
            if target_scale == 0:
                target_scale = 1.0
            observation_noise = gpr.kernel_.k2.noise_level * target_scale**2
            covariance -= observation_noise * np.eye(len(slice_grid))
            covariance = (covariance + covariance.T) / 2
            anchor_y = model_data.loc[anchor_index, target]
            anchor_position = np.flatnonzero(np.isclose(slice_grid, anchor_x))[0]
            anchor_variance = covariance[anchor_position, anchor_position]
            if anchor_variance <= 0:
                raise ValueError("Cannot condition the posterior slice at a zero-variance anchor.")
            anchor_covariance = covariance[:, anchor_position].copy()
            mean = mean + anchor_covariance * (
                (anchor_y - mean[anchor_position]) / anchor_variance
            )
            covariance -= np.outer(anchor_covariance, anchor_covariance) / anchor_variance
            covariance = (covariance + covariance.T) / 2
            eigenvalues, eigenvectors = np.linalg.eigh(covariance)
            covariance_factor = eigenvectors * np.sqrt(np.clip(eigenvalues, 0, None))
            posterior_samples = (
                mean
                + covariance_factor @ rng.standard_normal((len(slice_grid), 1))[:, 0]
            )
            posterior_std = np.sqrt(np.clip(np.diag(covariance), 0, None))
            axis.fill_between(
                slice_grid,
                mean - 1.96 * posterior_std,
                mean + 1.96 * posterior_std,
                color=color,
                alpha=0.10,
            )
            axis.plot(
                slice_grid,
                posterior_samples,
                color=color,
                alpha=1.0,
                linewidth=2.8,
                label=f"Posterior draw through sample {anchor_index}",
            )

            anchor_y = model_data.loc[anchor_index, target]
            axis.scatter(
                [anchor_x],
                [anchor_y],
                color="black",
                edgecolor="white",
                linewidth=0.8,
                marker="*",
                s=150,
                zorder=7,
                label="Measured anchor (conditioned)",
            )

        axis.scatter(
            X[feature],
            model_data[target],
            color="black",
            alpha=0.28,
            s=28,
            zorder=5,
            label="Other measured samples (ingredients vary)",
        )
        if row == 0:
            axis.set_title(target_labels[target])
        if col == 0:
            axis.set_ylabel(f"{feature}\n{target_labels[target]}")
        if feature != "TEM":
            axis.set_xlim(0, 6)
        if row == len(feature_cols) - 1:
            axis.set_xlabel(f"{feature} amount")
        axis.grid(alpha=0.2)
        if row == 0 and col == 0:
            axis.legend(loc="best", fontsize=8)

fig.suptitle("GPR 1D posterior slices with measured data in black", y=1.002)
fig.tight_layout()
plt.show()


In [32]:
# increase the length hyperparameters for the best model